# 🚀 ColabDrive Downloader & Merger
Công cụ tải video từ mọi nguồn (YouTube, Facebook, TikTok, Torrent/Magnet, Direct Link, luồng trực tiếp m3u8/mpd, danh sách phân mảnh .m4s/.ts) và ghép video tốc độ cao trực tiếp vào **Google Drive**.

**Đặc điểm nổi bật:**
- Tốc độ tải và xử lý cực nhanh nhờ hạ tầng mạng máy chủ Google Colab.
- Lưu trữ trực tiếp vào Google Drive để không bị mất dữ liệu khi ngắt kết nối.
- Giao diện Form trực quan dễ sử dụng, ẩn toàn bộ code phức tạp.
- Hỗ trợ tải phân mảnh video (m4s, ts...) từ danh sách link và tự động gộp chuẩn xác bằng `ffmpeg`.
- Hỗ trợ gộp ghép nhiều video khác nhau có sẵn trên Drive hoặc mới tải về.

**Hướng dẫn:** Chạy từng ô từ trên xuống dưới bằng nút ▶️ bên trái.


In [ ]:
#@title 📦 Bước 1: Cài đặt công cụ và thư viện nền tảng (Chạy 1 lần)
#@markdown Chạy ô này để tải và cài đặt các thư viện cần thiết (`yt-dlp`, `ffmpeg`, `aria2`).

import os
import sys

print("⏳ Đang cài đặt/cập nhật thư viện yt-dlp...")
os.system("pip install --force-reinstall -q yt-dlp")

print("⏳ Đang cài đặt aria2c và ffmpeg...")
os.system("apt-get update -y -q && apt-get install -y -q ffmpeg aria2")

print("✅ Đã cài đặt hoàn tất các công cụ!")


In [ ]:
#@title 💾 Bước 2: Kết nối Google Drive
#@markdown Chạy ô này để liên kết với tài khoản Google Drive của bạn. File tải về sẽ được lưu trữ trực tiếp trên Drive.

from google.colab import drive
import os

print("⏳ Vui lòng nhấn vào liên kết xuất hiện để cấp quyền truy cập Google Drive...")
drive.mount('/content/drive')

print("✅ Đã kết nối Google Drive thành công!")
print("📁 Thư mục gốc Drive của bạn: /content/drive/MyDrive/")


In [ ]:
#@title ⬇️ Bước 3: Tải & Ghép Video (Mọi Link / Mọi Nguồn)
#@markdown - **Nếu tải từ 1 Link chính (Youtube, Facebook, Torrent, m3u8, mpd, Direct...):** Dán link vào ô `LINK_VIDEO`. Hệ thống tự nhận diện và tải về.
#@markdown - **Nếu tải từ danh sách phân mảnh (.m4s, .ts):** Dán danh sách link (mỗi dòng 1 link) vào ô `LINK_VIDEO`. Điền link file khởi tạo vào ô `LINK_FILE_INIT` (nếu tải .m4s).

LINK_VIDEO = "" #@param {type:"string"}
LINK_FILE_INIT = "" #@param {type:"string"}
THU_MUC_DRIVE = "/content/drive/MyDrive/ColabDrive_Downloads" #@param {type:"string"}
TEN_FILE_LUU = "video.mp4" #@param {type:"string"}
HEADERS_TU_CHON = "" #@param {type:"string"}
TAI_NHANH_ARIA2 = True #@param {type:"boolean"}
SO_LUONG_TAI_SONG_SONG = 8 #@param {type:"slider", min:1, max:24, step:1}

import os
import re
import glob
import shutil
import urllib.request
import urllib.parse
import concurrent.futures
import yt_dlp
from tqdm.notebook import tqdm

def parse_headers(headers_str):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    if not headers_str.strip():
        return headers
    for line in headers_str.strip().split('\n'):
        if ':' in line:
            k, v = line.split(':', 1)
            headers[k.strip()] = v.strip()
    return headers

if not LINK_VIDEO.strip():
    raise ValueError("❌ Bạn chưa nhập link video!")

temp_dir = "/content/temp_download"
if os.path.exists(temp_dir):
    shutil.rmtree(temp_dir)
os.makedirs(temp_dir, exist_ok=True)
os.makedirs(THU_MUC_DRIVE, exist_ok=True)

headers = parse_headers(HEADERS_TU_CHON)
input_lines = [line.strip() for line in LINK_VIDEO.strip().split('\n') if line.strip()]

if len(input_lines) == 1:
    # --- TRƯỜNG HỢP 1: TẢI TỪ 1 LINK CHÍNH ---
    url = input_lines[0]
    print("🎬 Phát hiện tải từ 1 link chính. Đang xử lý...")
    
    is_torrent = url.startswith('magnet:?') or url.endswith('.torrent') or '.torrent?' in url
    if is_torrent:
        print("📦 Tải Torrent/Magnet...")
        trackers = ""
        try:
            import urllib.request
            req = urllib.request.Request('https://cf.trackerslist.com/best.txt', headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=5) as response:
                trackers_list = response.read().decode('utf-8').strip().split('\n')
                trackers = ','.join([t.strip() for t in trackers_list if t.strip()])
        except:
            pass
        
        cmd = f'aria2c --seed-time=0 --summary-interval=5 --bt-enable-lpd=true --enable-dht=true -d "{temp_dir}"'
        if trackers:
            cmd += f' --bt-tracker="{trackers}"'
        cmd += f' "{url}"'
        os.system(cmd)
        
        video_extensions = ('.mp4', '.mkv', '.avi', '.ts', '.mov', '.flv', '.webm', '.m4v')
        video_files = []
        for root, dirs, files_in_dir in os.walk(temp_dir):
            for f in files_in_dir:
                if f.lower().endswith(video_extensions):
                    full_path = os.path.join(root, f)
                    video_files.append((full_path, os.path.getsize(full_path)))
        if not video_files:
            raise Exception("❌ Không tìm thấy file video nào trong torrent!")
        video_files.sort(key=lambda x: x[1], reverse=True)
        FILE_PATH = video_files[0][0]
        file_name = os.path.basename(FILE_PATH)
        if TEN_FILE_LUU != "video.mp4":
            file_name = TEN_FILE_LUU
        dest_path = os.path.join(THU_MUC_DRIVE, file_name)
        shutil.move(FILE_PATH, dest_path)
        print(f"✅ Tải Torrent thành công! Đã lưu tại: {dest_path}")
    else:
        ydl_opts = {
            'outtmpl': os.path.join(temp_dir, '%(title)s.%(ext)s'),
            'format': 'bestvideo+bestaudio/best',
            'merge_output_format': 'mp4',
            'quiet': False,
            'no_warnings': False,
        }
        if headers:
            ydl_opts['http_headers'] = headers
        if TAI_NHANH_ARIA2:
            ydl_opts['external_downloader'] = 'aria2c'
            ydl_opts['external_downloader_args'] = ['-x16', '-s16', '-k1M']
            
        print("⬇️ Bắt đầu tải video...")
        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                ydl.download([url])
            downloaded = [f for f in glob.glob(os.path.join(temp_dir, '*')) if not f.endswith('.part') and not f.endswith('.ytdl')]
            if downloaded:
                FILE_PATH = downloaded[0]
                file_name = os.path.basename(FILE_PATH)
                if TEN_FILE_LUU != "video.mp4":
                    file_name = TEN_FILE_LUU
                dest_path = os.path.join(THU_MUC_DRIVE, file_name)
                shutil.move(FILE_PATH, dest_path)
                print(f"✅ Tải video thành công! Đã lưu tại: {dest_path}")
            else:
                raise Exception("Không tìm thấy file sau khi tải.")
        except Exception as e:
            print("⚠️ Thử cấu hình cơ bản không ghép kênh...")
            ydl_opts['format'] = 'best'
            try:
                with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                    ydl.download([url])
                downloaded = [f for f in glob.glob(os.path.join(temp_dir, '*')) if not f.endswith('.part') and not f.endswith('.ytdl')]
                FILE_PATH = downloaded[0]
                file_name = os.path.basename(FILE_PATH)
                if TEN_FILE_LUU != "video.mp4":
                    file_name = TEN_FILE_LUU
                dest_path = os.path.join(THU_MUC_DRIVE, file_name)
                shutil.move(FILE_PATH, dest_path)
                print(f"✅ Tải video thành công! Đã lưu tại: {dest_path}")
            except Exception as e2:
                raise Exception(f"❌ Tải thất bại! Chi tiết lỗi: {e2}")
else:
    # --- TRƯỜNG HỢP 2: TẢI & GHÉP DANH SÁCH PHÂN MẢNH ---
    print(f"🧩 Phát hiện danh sách {len(input_lines)} phân mảnh. Bắt đầu tải song song (Luồng: {SO_LUONG_TAI_SONG_SONG})...")
    
    def download_url(url, index, dest_dir):
        ext = "m4s"
        if ".ts" in url.lower() or "format=ts" in url.lower(): ext = "ts"
        elif ".mp4" in url.lower(): ext = "mp4"
        
        filename = f"segment_{index:06d}.{ext}" if index > 0 else f"init.{ext}"
        dest_file = os.path.join(dest_dir, filename)
        req = urllib.request.Request(url, headers=headers)
        try:
            with urllib.request.urlopen(req, timeout=30) as response, open(dest_file, 'wb') as f:
                f.write(response.read())
            return index, dest_file, True, None
        except Exception as e:
            return index, None, False, str(e)
            
    init_file_path = None
    if LINK_FILE_INIT.strip():
        print("⏳ Đang tải file khởi tạo INIT...")
        idx, path, success, err = download_url(LINK_FILE_INIT.strip(), 0, temp_dir)
        if success:
            print("✅ Tải file INIT thành công!")
            init_file_path = path
        else:
            print(f"⚠️ Không tải được file INIT ({err}). Tiếp tục tải các mảnh...")
            
    downloaded_paths = [None] * len(input_lines)
    failed_count = 0
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=SO_LUONG_TAI_SONG_SONG) as executor:
        futures = {executor.submit(download_url, url, i + 1, temp_dir): i for i, url in enumerate(input_lines)}
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(input_lines), desc="Tải phân mảnh"):
            idx = futures[future]
            try:
                index, path, success, err = future.result()
                if success: downloaded_paths[index - 1] = path
                else:
                    print(f"\n❌ Lỗi tải segment {index}: {err}")
                    failed_count += 1
            except Exception as e:
                print(f"\n❌ Lỗi tải segment {idx + 1}: {e}")
                failed_count += 1
                
    if failed_count > 0:
        print(f"\n⚠️ Tải lỗi {failed_count}/{len(input_lines)} mảnh. Tiếp tục gộp...")
    else:
        print("\n✅ Đã tải xong toàn bộ phân mảnh!")
        
    print("🔄 Đang gộp nhị phân...")
    raw_merged = "/content/temp_merged_raw.tmp"
    valid_paths = [p for p in downloaded_paths if p is not None]
    try:
        with open(raw_merged, 'wb') as outfile:
            if init_file_path and os.path.exists(init_file_path):
                with open(init_file_path, 'rb') as infile:
                    outfile.write(infile.read())
            for seg_path in valid_paths:
                with open(seg_path, 'rb') as infile:
                    outfile.write(infile.read())
        print("✅ Đã gộp thô nhị phân thành công.")
    except Exception as e:
        raise Exception(f"❌ Lỗi gộp nhị phân: {e}")
        
    print("🎬 Đang tối ưu container video bằng FFmpeg...")
    local_output = os.path.join("/content", TEN_FILE_LUU)
    if os.path.exists(local_output): os.remove(local_output)
    
    cmd = f'ffmpeg -i "{raw_merged}" -c copy -y "{local_output}" -v warning'
    status = os.system(cmd)
    
    if status == 0 and os.path.exists(local_output) and os.path.getsize(local_output) > 0:
        print("✅ FFmpeg tối ưu thành công!")
        final_src = local_output
    else:
        print("⚠️ FFmpeg tối ưu thất bại. Sử dụng file gộp thô trực tiếp...")
        final_src = raw_merged
        if not TEN_FILE_LUU.endswith(os.path.splitext(valid_paths[0])[1]):
            ext = os.path.splitext(valid_paths[0])[1]
            local_output = os.path.join("/content", os.path.splitext(TEN_FILE_LUU)[0] + ext)
            final_src = local_output
        os.rename(raw_merged, final_src)
        
    dest_drive = os.path.join(THU_MUC_DRIVE, os.path.basename(final_src))
    print(f"💾 Đang chuyển video sang Google Drive: {dest_drive}...")
    shutil.move(final_src, dest_drive)
    print("🎉 TẢI VÀ GHÉP THÀNH CÔNG!")
    print(f"📁 Đường dẫn trên Drive: {dest_drive}")

try:
    shutil.rmtree(temp_dir)
    if os.path.exists("/content/temp_merged_raw.tmp"): os.remove("/content/temp_merged_raw.tmp")
except: pass


In [ ]:
#@title 🎬 Bước 4: Ghép nhiều video có sẵn trên Drive / Colab
#@markdown Có 2 chế độ ghép:
#@markdown 1. **Ghép từ thư mục:** Ghép tất cả file video trong thư mục chỉ định (sắp xếp theo tên hoặc ngày tạo).
#@markdown 2. **Ghép theo danh sách:** Chỉ định chính xác các file cần ghép theo thứ tự mong muốn.

CHE_DO_GHEP = "Gh\u00e9p t\u1eeb th\u01b0 m\u1ee5c" #@param ["Gh\u00e9p t\u1eeb th\u01b0 m\u1ee5c", "Gh\u00e9p theo danh sách file tự chọn"]
DUONG_DAN_DAU_VAO = "/content/drive/MyDrive/ColabDrive_Downloads" #@param {type:"string"}
THU_MUC_DRIVE = "/content/drive/MyDrive/ColabDrive_Downloads" #@param {type:"string"}
TEN_FILE_LUU = "final_merged_video.mp4" #@param {type:"string"}
SAP_XEP_THEO = "T\u00ean A-Z" #@param ["T\u00ean A-Z", "Th\u1eddi gian tạo (cũ -> mới)", "Th\u1eddi gian tạo (mới -> cũ)"]
KIEU_GHEP = "Gh\u00e9p si\u00eau tốc (Không encode l\u1ea1i)" #@param ["Gh\u00e9p si\u00eau tốc (Không encode l\u1ea1i)", "Gh\u00e9p chất lượng (Có encode l\u1ea1i - tương th\u00edch cao)"]

import os
import glob
import shutil

# 1. Thu thập danh sách file
video_extensions = ('.mp4', '.mkv', '.avi', '.ts', '.mov', '.flv', '.webm', '.m4v')
files_to_merge = []

if CHE_DO_GHEP == "Ghép từ thư mục":
    if not os.path.exists(DUONG_DAN_DAU_VAO):
        raise FileNotFoundError(f"❌ Không tìm thấy thư mục đầu vào: {DUONG_DAN_DAU_VAO}")
    
    all_files = []
    for f in os.listdir(DUONG_DAN_DAU_VAO):
        full_path = os.path.join(DUONG_DAN_DAU_VAO, f)
        if os.path.isfile(full_path) and f.lower().endswith(video_extensions):
            all_files.append(full_path)
            
    if not all_files:
        raise ValueError(f"❌ Không tìm thấy video nào trong thư mục: {DUONG_DAN_DAU_VAO}")
        
    if SAP_XEP_THEO == "Tên A-Z":
        all_files.sort()
    elif SAP_XEP_THEO == "Thời gian tạo (cũ -> mới)":
        all_files.sort(key=os.path.getctime)
    elif SAP_XEP_THEO == "Thời gian tạo (mới -> cũ)":
        all_files.sort(key=os.path.getctime, reverse=True)
        
    files_to_merge = all_files
else:
    lines = [line.strip() for line in DUONG_DAN_DAU_VAO.strip().split('\n') if line.strip()]
    for line in lines:
        if os.path.exists(line):
            files_to_merge.append(line)
        else:
            print(f"⚠️ Cảnh báo: File không tồn tại và sẽ bị bỏ qua: {line}")
            
    if not files_to_merge:
        raise ValueError("❌ Không có file hợp lệ nào để tiến hành ghép!")

print(f"🎥 Tổng số file chuẩn bị ghép: {len(files_to_merge)}")
for i, f in enumerate(files_to_merge):
    print(f"  {i+1}. {os.path.basename(f)}")

# Tạo file danh sách cho FFmpeg
filelist_path = "/content/filelist.txt"
with open(filelist_path, 'w', encoding='utf-8') as f:
    for fp in files_to_merge:
        escaped_fp = fp.replace("'", "'\\''")
        f.write(f"file '{escaped_fp}'\n")

local_output_path = os.path.join("/content", TEN_FILE_LUU)
if os.path.exists(local_output_path):
    os.remove(local_output_path)

os.makedirs(THU_MUC_DRIVE, exist_ok=True)

# 2. Thực hiện ghép nối bằng FFmpeg
if KIEU_GHEP == "Ghép siêu tốc (Không encode lại)":
    print("\n⚡ Đang thực hiện ghép siêu tốc (chỉ copy stream, không encode lại)...")
    cmd = f'ffmpeg -f concat -safe 0 -i "{filelist_path}" -c copy -y "{local_output_path}" -v warning'
else:
    print("\n🎬 Đang thực hiện ghép có encode lại (sẽ chuyển đổi về H.264 + AAC)...")
    cmd = f'ffmpeg -f concat -safe 0 -i "{filelist_path}" -vf "scale=1920:1080:force_original_aspect_ratio=decrease,pad=1920:1080:(ow-iw)/2:(oh-ih)/2" -c:v libx264 -preset fast -crf 23 -c:a aac -b:a 192k -pix_fmt yuv420p -y "{local_output_path}" -v warning'

status = os.system(cmd)

if status == 0 and os.path.exists(local_output_path) and os.path.getsize(local_output_path) > 0:
    dest_drive_path = os.path.join(THU_MUC_DRIVE, TEN_FILE_LUU)
    print(f"💾 Đang di chuyển file đã ghép lên Google Drive: {dest_drive_path}...")
    shutil.move(local_output_path, dest_drive_path)
    print("🎉 HOÀN THÀNH GHÉP VIDEO!")
    print(f"📁 File đã lưu tại: {dest_drive_path}")
else:
    print("❌ Lỗi xảy ra trong quá trình ghép video bằng FFmpeg.")
    if os.path.exists(local_output_path):
        try: os.remove(local_output_path)
        except: pass

if os.path.exists(filelist_path):
    os.remove(filelist_path)
